# Preço / Lucro Histórico de Empresas Listadas na B3 com Dados da CVM

O preço sobre lucro é um indicador fundamentalista para análise de empresas, com ele sabemos o tempo de retorno do investimento, em teoria quanto menor melhor.

Neste estudo vamos coletar os dados da DRE (Demonstrativo do Resultado do Exercício) de empresas listadas na B3, os dados serão coletados diretamente do site da CVM.

## Dependências e Configurações

In [1]:
from zipfile import ZipFile
import tempfile
import requests
from pathlib import Path
from loguru import logger

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

logger.info('BIBLIOTECAS CARREGADAS')

2026-06-10 19:41:38.932 | INFO     | __main__:<module>:12 - BIBLIOTECAS CARREGADAS


In [ ]:
DATA            = Path().cwd()/'dados'
DATA.mkdir(parents=True, exist_ok=True)
logger.info('DIRETORIO PARA DADOS CRIADO')


URL_BASE        = 'https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/'
ANO_INICIO      = 2010
ANO_FIM         = 2025
DEMONSTRATIVO   = 'DRE_con'
ARQUIVOS_ZIP    = [ f'dfp_cia_aberta_{ano}.zip' for ano in range(ANO_INICIO, ANO_FIM + 1) ]

logger.info('CONSTANTES CARREGADAS')

2026-06-10 20:14:52.041 | INFO     | __main__:<module>:3 - DIRETORIO PARA DADOS CRIADO
2026-06-10 20:14:52.052 | INFO     | __main__:<module>:14 - CONSTANTES CARREGADAS


## 1. Coleta de Arquivos na CVM

In [10]:
logger.info('Iniciando download e extração de arquivos CSVs')
logger.info(f'Total de arquivos em formato zip para download: {len(ARQUIVOS_ZIP)}')

# temp
with tempfile.TemporaryDirectory() as temp:
    temp_path = Path(temp)
    logger.debug(f'Pasta temporária criada em: {temp_path}')

    # loop para baixar e extrair 
    for arquivo in ARQUIVOS_ZIP:
        url_completa = URL_BASE + arquivo
        caminho_zip = temp_path / arquivo
        
        try:            
            logger.info(f'Baixando: {arquivo}')                             
            resposta = requests.get(url_completa, stream=True)       
            resposta.raise_for_status() 
            
            # arq em modo de escrita binária ('wb') e salva em pedaços (chunks)
            with open(caminho_zip, 'wb') as f:
                for chunk in resposta.iter_content(chunk_size=8192): 
                    if chunk: 
                        f.write(chunk)
            
            # extracao
            logger.info(f'Extraindo {arquivo}...')
            with ZipFile(caminho_zip, 'r') as zip_ref:
                zip_ref.extractall(DATA)
                
        except requests.exceptions.HTTPError as http_err:            
            logger.exception(f'Erro HTTP ao tentar baixar {arquivo}')
        except Exception as e:            
            logger.exception(f'Erro genérico ao processar {arquivo}')

logger.debug(f'Arquivos extraídos em: {DATA}')
logger.info('Processo Concluído com sucesso')

2026-06-10 19:56:34.363 | INFO     | __main__:<module>:1 - Iniciando download e extração de arquivos CSVs
2026-06-10 19:56:34.364 | INFO     | __main__:<module>:2 - Total de arquivos em formato zip para download: 16
2026-06-10 19:56:34.367 | DEBUG    | __main__:<module>:7 - Pasta temporária criada em: C:\Users\wsant\AppData\Local\Temp\tmp9o3etche
2026-06-10 19:56:34.367 | INFO     | __main__:<module>:15 - Baixando: dfp_cia_aberta_2010.zip
2026-06-10 19:56:35.182 | INFO     | __main__:<module>:26 - Extraindo dfp_cia_aberta_2010.zip...
2026-06-10 19:56:37.850 | INFO     | __main__:<module>:15 - Baixando: dfp_cia_aberta_2011.zip
2026-06-10 19:56:38.418 | INFO     | __main__:<module>:26 - Extraindo dfp_cia_aberta_2011.zip...
2026-06-10 19:56:40.600 | INFO     | __main__:<module>:15 - Baixando: dfp_cia_aberta_2012.zip
2026-06-10 19:56:41.188 | INFO     | __main__:<module>:26 - Extraindo dfp_cia_aberta_2012.zip...
2026-06-10 19:56:43.296 | INFO     | __main__:<module>:15 - Baixando: dfp_cia_

## 2. Coletando DRE

In [13]:
logger.info('Buscando arquivos CSV com DRE')

arquivos_csv     = list(DATA.glob('*.csv'))
arquivos_csv_dre = [arquivo for arquivo in arquivos_csv if DEMONSTRATIVO in arquivo.name]

if not arquivos_csv_dre:
    logger.warning('Nenhum arquivo csv com DRE encontrado.')

logger.info(f'Total de arquivos encontrados: {len(arquivos_csv_dre)}')

logger.info('Criando lista de dataframes')
lista_df = [
    pd.read_csv(arquivo, sep=';', encoding='ISO-8859-1', decimal=',', dtype=str) # dtype str :: evitar erro de tipos no concat
        for arquivo in arquivos_csv_dre
]

logger.info('Iniciar Concatenação')
df = pd.concat(lista_df, ignore_index=True)
logger.info(f'DataFrame concatenado, o dataset possui: {df.shape[0]} linhas e {df.shape[1]} colunas.')

2026-06-10 20:19:12.350 | INFO     | __main__:<module>:1 - Buscando arquivos CSV com DRE
2026-06-10 20:19:12.357 | INFO     | __main__:<module>:9 - Total de arquivos encontrados: 16
2026-06-10 20:19:12.359 | INFO     | __main__:<module>:11 - Criando lista de dataframes
2026-06-10 20:19:16.213 | INFO     | __main__:<module>:17 - Iniciar Concatenação
2026-06-10 20:19:16.239 | INFO     | __main__:<module>:19 - DataFrame concatenado, o dataset possui: 449570 linhas e 15 colunas.


In [14]:
df.head()

,CNPJ_CIA,DT_REFER,VERSAO,DENOM_CIA,CD_CVM,GRUPO_DFP,MOEDA,ESCALA_MOEDA,ORDEM_EXERC,DT_INI_EXERC,DT_FIM_EXERC,CD_CONTA,DS_CONTA,VL_CONTA,ST_CONTA_FIXA
0,00.000.000/0001-91,2010-12-31,3,BCO BRASIL S.A.,001023,DF Consolidado - Demonstração do Resultado,REAL,MIL,PENÚLTIMO,2009-01-01,2009-12-31,3.01,Receitas da Intermediação Financeira,67608506.0000000000,S
1,00.000.000/0001-91,2010-12-31,3,BCO BRASIL S.A.,001023,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2010-01-01,2010-12-31,3.01,Receitas da Intermediação Financeira,85143206.0000000000,S
2,00.000.000/0001-91,2010-12-31,3,BCO BRASIL S.A.,001023,DF Consolidado - Demonstração do Resultado,REAL,MIL,PENÚLTIMO,2009-01-01,2009-12-31,3.01.01,Receita de Juros,67608506.0000000000,S
3,00.000.000/0001-91,2010-12-31,3,BCO BRASIL S.A.,001023,DF Consolidado - Demonstração do Resultado,REAL,MIL,ÚLTIMO,2010-01-01,2010-12-31,3.01.01,Receita de Juros,85143206.0000000000,S
4,00.000.000/0001-91,2010-12-31,3,BCO BRASIL S.A.,001023,DF Consolidado - Demonstração do Resultado,REAL,MIL,PENÚLTIMO,2009-01-01,2009-12-31,3.02,Despesas da Intermediação Financeira,-39302642.0000000000,S
